# Exercise 01: Data Exploration — Solution

This is the completed reference solution. Your code may differ — that's fine as long as you addressed the data quality issues and explored the key patterns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

DATA_DIR = "../../../../resources/datasets"

## Part 1: Load and Inspect the Car Sales Data

In [ ]:
car_sales = pd.read_csv(f"{DATA_DIR}/car-sales-extended-missing-data.csv")
print(f"Shape: {car_sales.shape}")
print(f"\nColumn types:\n{car_sales.dtypes}")

In [ ]:
car_sales.head()

In [ ]:
car_sales.describe()

In [ ]:
print("Missing values per column:")
print(car_sales.isnull().sum())
print(f"\nTotal rows: {len(car_sales)}")
print(f"Rows with any missing value: {car_sales.isnull().any(axis=1).sum()}")

## Part 2: Handle Missing Data

Strategy for each column:
- **Make** (categorical): Fill with mode — the most common make
- **Colour** (categorical): Fill with mode
- **Odometer** (numerical): Fill with median (more robust to outliers than mean)
- **Doors** (numerical but really categorical — 3, 4, or 5): Fill with mode (4 is most common)
- **Price** (target): Drop rows where Price is missing — we can't learn from examples without a target

In [ ]:
# Drop rows where target (Price) is missing
car_sales = car_sales.dropna(subset=["Price"])

# Fill categorical columns with mode
car_sales["Make"] = car_sales["Make"].fillna(car_sales["Make"].mode()[0])
car_sales["Colour"] = car_sales["Colour"].fillna(car_sales["Colour"].mode()[0])

# Fill numerical columns
car_sales["Odometer (KM)"] = car_sales["Odometer (KM)"].fillna(car_sales["Odometer (KM)"].median())
car_sales["Doors"] = car_sales["Doors"].fillna(car_sales["Doors"].mode()[0])

print("Missing values after cleaning:")
print(car_sales.isnull().sum())
print(f"\nRows remaining: {len(car_sales)}")

## Part 3: Explore Distributions

In [ ]:
# Price distribution
plt.figure(figsize=(10, 5))
plt.hist(car_sales["Price"], bins=30, edgecolor="black", alpha=0.7)
plt.xlabel("Price ($)")
plt.ylabel("Count")
plt.title("Distribution of Car Sale Prices")
plt.axvline(car_sales["Price"].mean(), color="red", linestyle="--", label=f"Mean: ${car_sales['Price'].mean():,.0f}")
plt.axvline(car_sales["Price"].median(), color="green", linestyle="--", label=f"Median: ${car_sales['Price'].median():,.0f}")
plt.legend()
plt.show()

In [ ]:
# Scatter plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(car_sales["Odometer (KM)"], car_sales["Price"], alpha=0.5)
axes[0].set_xlabel("Odometer (KM)")
axes[0].set_ylabel("Price ($)")
axes[0].set_title("Odometer vs Price")

axes[1].scatter(car_sales["Doors"], car_sales["Price"], alpha=0.5)
axes[1].set_xlabel("Doors")
axes[1].set_ylabel("Price ($)")
axes[1].set_title("Doors vs Price")

plt.tight_layout()
plt.show()

# Odometer shows a clear negative relationship — higher mileage = lower price.
# Doors shows no obvious pattern — probably not a strong predictor.

In [ ]:
# Correlation heatmap
numeric_cols = car_sales.select_dtypes(include=[np.number])
plt.figure(figsize=(8, 6))
sns.heatmap(numeric_cols.corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation Heatmap — Car Sales")
plt.show()

# Odometer and Price have a moderate negative correlation.
# Doors has weak correlation with Price.

In [ ]:
# Categorical value counts
print("Make distribution:")
print(car_sales["Make"].value_counts())
print(f"\nColour distribution:")
print(car_sales["Colour"].value_counts())

## Part 4: Explore the Heart Disease Data

In [ ]:
heart = pd.read_csv(f"{DATA_DIR}/heart-disease.csv")
print(f"Shape: {heart.shape}")
print(f"\nMissing values: {heart.isnull().sum().sum()}")
heart.head()

In [ ]:
heart.describe()

In [ ]:
# Class balance
print("Target distribution:")
print(heart["target"].value_counts())
print(f"\nDisease rate: {heart['target'].mean():.1%}")

# This is fairly balanced (~54% positive, ~46% negative).
# With balanced classes, accuracy is a reasonable metric.
# If it were 95/5, we'd need to be much more careful about metrics.

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(heart.corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f", square=True)
plt.title("Correlation Heatmap — Heart Disease")
plt.tight_layout()
plt.show()

# Strongest correlations with target:
# - cp (chest pain type): positive correlation
# - thalach (max heart rate): positive correlation
# - exang (exercise-induced angina): negative correlation
# - oldpeak (ST depression): negative correlation
# - ca (number of vessels): negative correlation

In [ ]:
# Visualize promising features
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.boxplot(x="target", y="thalach", data=heart, ax=axes[0])
axes[0].set_title("Max Heart Rate by Disease Status")
axes[0].set_xlabel("Heart Disease (0=No, 1=Yes)")

sns.boxplot(x="target", y="oldpeak", data=heart, ax=axes[1])
axes[1].set_title("ST Depression by Disease Status")
axes[1].set_xlabel("Heart Disease (0=No, 1=Yes)")

sns.countplot(x="cp", hue="target", data=heart, ax=axes[2])
axes[2].set_title("Chest Pain Type by Disease Status")
axes[2].set_xlabel("Chest Pain Type")

plt.tight_layout()
plt.show()

# Patients WITH heart disease tend to have:
# - Higher max heart rate (thalach)
# - Lower ST depression (oldpeak)
# - Different chest pain type distribution

## Reflection

1. **Biggest data quality issue:** The car sales data had missing values in every column, including the target (Price). The most important decision was dropping rows where Price was missing — you can't train a model without the thing you're trying to predict. For the feature columns, filling with median/mode was reasonable since the missing values appeared random.

2. **Is the heart disease dataset balanced?** Yes — roughly 54/46 split. If it were heavily imbalanced (e.g., 95/5), accuracy would be misleading. A model that always predicts "no disease" would be 95% accurate but completely useless. With balanced data, accuracy is a reasonable starting metric.

3. **Surprise:** The Doors column in the car sales data has almost no relationship with Price. Intuitively you might expect 4-door sedans vs 2-door sports cars to differ, but the data doesn't show it. This is a good reminder: check your assumptions with data.

4. **Regression vs classification exploration:** With regression data, you look at distributions, scatter plots, and correlations — all continuous relationships. With classification data, you focus more on class balance, group comparisons (box plots), and how features differ between classes. The questions you ask are fundamentally different: "what's the relationship?" (regression) vs "what separates the groups?" (classification).